# Train Intent TextCNN (CPU, TensorFlow)

Vài phút trên CPU, chạy bằng `.venv-tf`. Có thể chạy trong lúc GPU đang train detection.
Kết quả: `intent_int8.tflite`, `vocab.json`, `summary.json` trong thư mục run.

Bản v2 (14/11): augmentation tạo câu "khuôn trong phạm vi + vật ngoài 7 lớp" với nhãn OUT_OF_SCOPE.
Ghi vào `runs/intent_v2`, không đụng `runs/intent` cũ.

In [ ]:
import sys, json, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "nbrun.py").exists() else Path.cwd() / "notebooks"))
import nbrun; importlib.reload(nbrun)
from nbrun import ROOT, PIPE, PY_TORCH, PY_TF, run, check_gpu, show_json, show_log, backup
print("Gốc dự án:", ROOT)

In [ ]:
# ---- Cấu hình intent: chỉ cần sửa ở đây ----
INTENT_OUT    = "runs/intent_v2"
INTENT_EPOCHS = 40
INTENT_FOLDS  = 5
DET_FOR_TEST  = "runs/det_torch_base/det_int8.tflite"   # detection dùng cho file nộp thử (mô hình 160x160)
SUB_TEST      = "submission_intent_v2.csv"

## 1. Train

In [ ]:
if (ROOT / INTENT_OUT / "summary.json").exists():
    print(f"CHÚ Ý: {INTENT_OUT} đã có kết quả và sẽ bị ghi đè.")
run(PY_TF, [PIPE / "intent/train_intent.py", "--data", ".", "--out", INTENT_OUT,
            "--epochs", INTENT_EPOCHS, "--folds", INTENT_FOLDS], backend="tensorflow")

## 2. Kết quả
- `cv_acc`: accuracy cross-validation theo khuôn câu (cần ≥ 0,998).
- `cv_comp_count` / `cv_comp_exist` / `cv_comp_location`: tỉ lệ câu tổ hợp (khuôn trong phạm vi + vật ngoài phạm vi) được đoán đúng là OUT_OF_SCOPE (cần ≥ 0,98; bản cũ: 0,96 / 0,86 / 0,01).
- `fold0_float` so với `fold0_int8`: lệch không quá 0,002.

In [ ]:
show_json(f"{INTENT_OUT}/summary.json");

In [ ]:
s = json.load(open(ROOT / INTENT_OUT / "summary.json"))
old = json.load(open(ROOT / "runs/intent/summary.json"))
checks = {
    "cv_acc >= 0.998": s["cv_acc"] >= 0.998,
    **{f"{k} >= 0.98": (s.get(k) or 0) >= 0.98 for k in ("cv_comp_count", "cv_comp_exist", "cv_comp_location")},
    "|fold0_int8 - fold0_float| <= 0.002": abs(s["fold0_int8"] - s["fold0_float"]) <= 0.002,
    "params == 28005": s["params"] == 28005,
}
for k, ok in checks.items():
    print(("ĐẠT  " if ok else "CHƯA ĐẠT  ") + k)
print("Toán tử TFLite:", s.get("ops"))

## 3. File nộp thử: intent v2 + detection cũ
Ghi ra `submission_intent_v2.csv` (không đè `submission.csv`). Dòng "Phân bố intent dự đoán trên test" mong đợi mỗi nhãn gần 20%.
Nếu OUT_OF_SCOPE vẫn dưới 19% thì dừng và báo lại; **không mở từng câu test ra xem**.

In [ ]:
import pandas as pd
run(PY_TF, [PIPE / "make_submission.py", "--data", ".", "--det", ROOT / DET_FOR_TEST,
            "--intent", ROOT / INTENT_OUT / "intent_int8.tflite", "--vocab", ROOT / INTENT_OUT / "vocab.json",
            "--out", SUB_TEST], backend="tensorflow", imgsz="160")   # DET_FOR_TEST là mô hình 160x160
sub = pd.read_csv(ROOT / SUB_TEST)
dist = sub[sub.task_type == "INTENT"].prediction.value_counts(normalize=True)
print(dist.round(3).to_dict())
print("OUT_OF_SCOPE", f"{dist.get('OUT_OF_SCOPE', 0):.1%}", "-> ĐẠT" if dist.get("OUT_OF_SCOPE", 0) >= 0.19 else "-> DƯỚI 19%: dừng, báo lại")